# 12 — ISIC 2019 Prep + Dengeli Veri Seti İnşası

ISIC 2019'dan **hem melanoma hem non-melanoma** görüntülerini indirir,
HAM10000 ile aynı önişleme pipeline'ını uygular ve **dengeli, çok-kaynaklı**
ikili veri setini (`X_combined.npy` + dengeli lesion-grouped split) Drive'a yazar.

Dengeli set tasarımı (kaynak ⊥ etiket → shortcut yok):
- **melanoma**     = tüm HAM mel (1.113) + tüm ISIC mel (~4.522)
- **non-melanoma** = HAM non-mel (1.113, mel sayısına eşit) + ISIC non-mel (~4.522)
- Toplam ≈ **11.270, tam 1:1 dengeli**, train/val/test hepsi dengeli.

**Bunu BİR KERE çalıştır. Sonra 04→09 CNN'leri A100'de eğit.**

- Çıktı: `MyDrive/melanoma/data/X_combined.npy` (~1.7 GB) + idx_*_bal.npy
- Süre: ISIC indir ~15 dk (ilk sefer) + önişleme ~12 dk + inşa ~3 dk
- Runtime: **T4 GPU** (iş CPU-bound; A100 israfı olur)

In [1]:
# --- Setup ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)
else:
    subprocess.run(["git", "-C", str(project_root), "pull"], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import config
config.ensure_drive_dirs()
print("OK — Data dir:", config.DATA_DIR)

Mounted at /content/drive
OK — Data dir: /content/drive/MyDrive/melanoma/data


In [2]:
# --- X_all.npy'nin gerçek boyutunu oku (224 mı 448 mi?) ---
import numpy as np
from pathlib import Path

x_path = config.DATA_DIR / "X_all.npy"
X_tmp = np.load(x_path, mmap_mode='r')
ACTUAL_SIZE = X_tmp.shape[1]   # 224 veya 448
print(f"X_all.npy shape: {X_tmp.shape}")
print(f"ISIC 2019 da bu boyuta ({ACTUAL_SIZE}px) önişlenecek")

X_all.npy shape: (10015, 224, 224, 3)
ISIC 2019 da bu boyuta (224px) önişlenecek


In [3]:
# --- Kaggle auth ---
from pathlib import Path

kaggle_json = Path("/root/.kaggle/kaggle.json")
if not kaggle_json.exists():
    from google.colab import files
    print("kaggle.json yükle (kaggle.com → sağ üst profil → Settings → API → Create New Token)")
    uploaded = files.upload()
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.move(list(uploaded.keys())[0], str(kaggle_json))
    kaggle_json.chmod(0o600)
    print("Tamam.")
else:
    print("kaggle.json zaten var.")

kaggle.json yükle (kaggle.com → sağ üst profil → Settings → API → Create New Token)


Saving kaggle (1).json to kaggle (1).json
Tamam.


In [4]:
# --- ISIC 2019 indir (~10 GB, ~15 dk) ---
import subprocess
from pathlib import Path

ISIC_DIR = Path("/content/isic2019")
ISIC_DIR.mkdir(exist_ok=True)

csv_path = ISIC_DIR / "ISIC_2019_Training_GroundTruth.csv"
if not csv_path.exists():
    print("İndiriliyor (~10 GB, ~15 dk bekliyorsun)...")
    result = subprocess.run(
        ["kaggle", "datasets", "download",
         "-d", "andrewmvd/isic-2019",
         "-p", str(ISIC_DIR), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("İndirme başarısız — kaggle.json doğru mu?")
    print("İndirme tamamlandı.")
else:
    print("Zaten indirilmiş.")

print("Dosyalar:", [f.name for f in sorted(ISIC_DIR.iterdir())[:10]])

İndiriliyor (~10 GB, ~15 dk bekliyorsun)...
İndirme tamamlandı.
Dosyalar: ['ISIC_2019_Training_GroundTruth.csv', 'ISIC_2019_Training_Input', 'ISIC_2019_Training_Metadata.csv']


In [ ]:
# --- MEL listesi + stratified NON-MEL örneklemi (MEL ile eşit sayıda) ---
import pandas as pd, os, numpy as np
from pathlib import Path

gt = pd.read_csv(csv_path)
# ISIC 2019 GT sütunları: image, MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK
NONMEL_COLS = [c for c in ["NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"] if c in gt.columns]

mel_ids = gt[gt["MEL"] == 1]["image"].tolist()
print(f"Toplam MEL: {len(mel_ids)}")

# Resim klasörünü bul
IMG_DIR = None
for candidate in [ISIC_DIR / "ISIC_2019_Training_Input", ISIC_DIR / "train", ISIC_DIR]:
    if candidate.exists() and any(candidate.glob("*.jpg")):
        IMG_DIR = candidate
        break
    for sub in candidate.glob("*/"):
        if any(sub.glob("*.jpg")):
            IMG_DIR = sub
            break
    if IMG_DIR:
        break
if IMG_DIR is None:
    for root, dirs, files in os.walk(str(ISIC_DIR)):
        if any(f.endswith(".jpg") for f in files):
            IMG_DIR = Path(root)
            break
print("Resim klasörü:", IMG_DIR)

# NON-MEL'i sınıflarına göre stratified örnekle (MEL sayısı kadar)
n_target = len(mel_ids)
nonmel_df = gt[gt["MEL"] == 0].copy()

def _row_class(r):
    for c in NONMEL_COLS:
        if r[c] == 1:
            return c
    return "OTHER"

nonmel_df["cls"] = nonmel_df.apply(_row_class, axis=1)
nonmel_df = nonmel_df[nonmel_df["cls"] != "OTHER"]

picked = []
for c, grp in nonmel_df.groupby("cls"):
    share = int(round(n_target * len(grp) / len(nonmel_df)))
    picked += grp.sample(n=min(share, len(grp)), random_state=42)["image"].tolist()

rng = np.random.default_rng(42)
picked = list(dict.fromkeys(picked))                      # uniq
if len(picked) > n_target:
    picked = list(rng.permutation(picked))[:n_target]
elif len(picked) < n_target:
    extra = [i for i in nonmel_df["image"].tolist() if i not in set(picked)]
    picked += list(rng.permutation(extra))[:(n_target - len(picked))]
nonmel_ids = picked
print(f"Seçilen NON-MEL (stratified): {len(nonmel_ids)}")

mel_paths    = [(i, IMG_DIR / f"{i}.jpg") for i in mel_ids    if (IMG_DIR / f"{i}.jpg").exists()]
nonmel_paths = [(i, IMG_DIR / f"{i}.jpg") for i in nonmel_ids if (IMG_DIR / f"{i}.jpg").exists()]
print(f"MEL bulundu: {len(mel_paths)}/{len(mel_ids)}  |  NON-MEL bulundu: {len(nonmel_paths)}/{len(nonmel_ids)}")

In [ ]:
# --- Paralel önişleme: MEL + NON-MEL (4 thread, ~10-12 dk) ---
import cv2, numpy as np, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from src.preprocessing import preprocess_for_storage

N_WORKERS = 4
SIZE = ACTUAL_SIZE   # HAM10000 ile aynı boyut (224)

def _preprocess_list(pairs, tag):
    X = np.zeros((len(pairs), SIZE, SIZE, 3), dtype=np.uint8)
    ids = np.empty(len(pairs), dtype=object)

    def work(args):
        idx, img_id, img_path = args
        img = cv2.imread(str(img_path))
        if img is None:
            return idx, img_id, np.zeros((SIZE, SIZE, 3), np.uint8)
        try:
            rgb, _ = preprocess_for_storage(img, size=SIZE)
            return idx, img_id, rgb
        except Exception:
            return idx, img_id, np.zeros((SIZE, SIZE, 3), np.uint8)

    tasks = [(i, img_id, p) for i, (img_id, p) in enumerate(pairs)]
    t0, n_done = time.time(), 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futs = {pool.submit(work, t): t[0] for t in tasks}
        for fut in as_completed(futs):
            idx, img_id, rgb = fut.result()
            X[idx], ids[idx] = rgb, img_id
            n_done += 1
            if n_done % 500 == 0:
                el = time.time() - t0
                print(f"  [{tag}] {n_done}/{len(pairs)}  {n_done/el:.1f} img/s")
    print(f"  [{tag}] bitti: {len(pairs)} görüntü, {(time.time()-t0)/60:.1f} dk")
    return X, ids

X_mel, ids_mel = _preprocess_list(mel_paths, "MEL")
X_non, ids_non = _preprocess_list(nonmel_paths, "NON-MEL")
print(f"\nX_mel: {X_mel.shape}  |  X_non: {X_non.shape}")

In [ ]:
# --- Drive'a kaydet (mel + non-mel) ---
import numpy as np

np.save(config.DATA_DIR / "X_isic2019_mel.npy",      X_mel)
np.save(config.DATA_DIR / "ids_isic2019_mel.npy",    ids_mel)
np.save(config.DATA_DIR / "X_isic2019_nonmel.npy",   X_non)
np.save(config.DATA_DIR / "ids_isic2019_nonmel.npy", ids_non)

print("Kaydedildi:")
for f in ["X_isic2019_mel.npy", "X_isic2019_nonmel.npy"]:
    print(f"  {f}: {(config.DATA_DIR / f).stat().st_size/1e6:.0f} MB")

# RAM'i serbest bırak (bir sonraki hücre diskten yeniden yükleyecek)
del X_mel, X_non, ids_mel, ids_non
import gc; gc.collect()

In [ ]:
# --- Dengeli, çok-kaynaklı veri setini inşa et + Drive'a yaz ---
import importlib, src.data as _sd
importlib.reload(_sd)   # module cache sorununu önler
from src.data import build_balanced_dataset

build_balanced_dataset(config.DATA_DIR, seed=config.SEED, save=True, verbose=True)

# Yazılan dosyaları doğrula
need = ["X_combined.npy", "y_combined.npy", "ids_combined.npy", "source_combined.npy",
        "lesion_combined.npy", "idx_train_bal.npy", "idx_val_bal.npy", "idx_test_bal.npy"]
print("\n=== Drive'a yazılan dosyalar ===")
ok = True
for f in need:
    p = config.DATA_DIR / f
    if p.exists():
        print(f"  OK    {f}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  EKSİK {f}")
        ok = False

print("\n" + ("HAZIR. Şimdi 04_alexnet.ipynb'i A100'de çalıştır (doğrulama adımı)."
              if ok else "!!! Bazı dosyalar yazılamadı — yukarıdaki hatalara bak."))